# Submission v11 — v7c features + LightGBM+CatBoost blend (ISOLATED)

## Single isolated change vs v7c
- Same v7c features (106 features: absolute stats + delta/slope/t1t3 + HRV time-domain + pid_enc + accel_mag)
- Same class weights (capped at 2.5 for class 1)
- Same 3 seeds × 5 folds Stratified CV
- Same calibration α=1.8

**Only change:** add CatBoost trained on identical features. Output 3 submissions:
1. `v11_lgb` — LGB only (sanity check, should reproduce v7c LOPO=0.5130)
2. `v11_cat` — CatBoost only
3. `v11_blend` — equal-weight average of LGB and CAT raw probabilities

NO lag features. NO SMOTE. Pure isolation of the blend effect.

## Decision rule
- If `v11_lgb` LOPO matches v7c (~0.5130) → sanity check passed
- If `v11_blend` LOPO > 0.5130 → blend is the real fix, submit blend
- If `v11_blend` LOPO ≤ v7c → blend doesn't help, stick with v7c


In [1]:
%pip install lightgbm catboost scikit-learn pandas numpy scipy -q


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from catboost import CatBoostClassifier
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


In [3]:
SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']
WINDOW_MS = 180_000
HALF_MS   =  90_000
THIRD_MS  =  60_000

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']:
            f['hrv_'+k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff)>0 else 0.
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff)>25))*100 if len(rr_diff)>0 else 0.
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff)>50))*100 if len(rr_diff)>0 else 0.
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn']/f['hrv_mean_rr'] if f['hrv_mean_rr']>1e-6 else 0.
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']; ts = float(lrow['timestamp']); lid = lrow['id']
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None: rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<=ts),           SENSOR_COLS]
        wf  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-HALF_MS),    SENSOR_COLS]
        wl  = sg.loc[(ta>=ts-HALF_MS)  &(ta<=ts),           SENSOR_COLS]
        wt1 = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-2*THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta>=ts-THIRD_MS) &(ta<=ts),           SENSOR_COLS]
        for col in SENSOR_COLS:
            v   = wa[col].dropna().values.astype(float)
            vf  = wf[col].dropna().values.astype(float)
            vl  = wl[col].dropna().values.astype(float)
            vt1 = wt1[col].dropna().values.astype(float)
            vt3 = wt3[col].dropna().values.astype(float)
            if len(v)==0:
                for s in ['mean','std','min','max','median','skew','kurt','range','q25','q75','iqr',
                          'delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[col+'_'+s] = np.nan
                continue
            feat[col+'_mean']   = float(np.mean(v))
            feat[col+'_std']    = float(np.std(v))
            feat[col+'_min']    = float(np.min(v))
            feat[col+'_max']    = float(np.max(v))
            feat[col+'_median'] = float(np.median(v))
            feat[col+'_skew']   = float(spstats.skew(v)) if len(v)>2 else 0.
            feat[col+'_kurt']   = float(spstats.kurtosis(v)) if len(v)>2 else 0.
            feat[col+'_range']  = float(np.max(v)-np.min(v))
            feat[col+'_q25']    = float(np.percentile(v,25))
            feat[col+'_q75']    = float(np.percentile(v,75))
            feat[col+'_iqr']    = float(np.percentile(v,75)-np.percentile(v,25))
            feat[col+'_delta']  = float(np.mean(vl)-np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.
            feat[col+'_slope']  = float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.
            feat[col+'_t1_mean'] = float(np.mean(vt1)) if len(vt1)>0 else 0.
            feat[col+'_t3_mean'] = float(np.mean(vt3)) if len(vt3)>0 else 0.
            feat[col+'_t3t1']    = feat[col+'_t3_mean']-feat[col+'_t1_mean']

        if len(wa) > 0:
            ax = wa['accel_x'].fillna(0).values; ay = wa['accel_y'].fillna(0).values; az = wa['accel_z'].fillna(0).values
            mag = np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean']=feat['accel_mag_std']=feat['accel_mag_max']=np.nan

        feat.update(hrv_time_domain(wa['heart_rate']))
        feat['pid_enc'] = pid_enc_map.get(pid, -1)
        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p:i for i,p in enumerate(TRAIN_LABEL['pid'].unique())}

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
print(f'  train shape: {train_features.shape}')

print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, train_pid_map)
print(f'  test shape: {test_features.shape}')


Extracting train features...
  train shape: (815, 106)
Extracting test features...
  test shape: (1028, 106)


In [4]:
tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),
                           columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),
                           columns=test_features.columns, index=test_features.index)

counts        = Counter(y)
total         = len(y)
n_cls         = len(counts)
class_weights = {0: total/(n_cls*counts[0]),
                 1: min(total/(n_cls*counts[1]), 2.5),
                 2: total/(n_cls*counts[2])}
sample_weights = np.array([class_weights[yi] for yi in y])
train_prior    = np.array([counts[i]/total for i in range(3)])

print('X_imp shape     :', X_imp.shape)
print('X_test_imp shape:', X_test_imp.shape)
print('Class weights (capped):', {k: round(v,3) for k,v in class_weights.items()})


X_imp shape     : (815, 106)
X_test_imp shape: (1028, 106)
Class weights (capped): {0: 1.677, 1: 2.5, 2: 0.463}


## LOPO CV — LGB / CAT / BLEND on identical v7c features

No SMOTE, no lag features. The only difference between this and v7c is the addition of CatBoost as a second learner.

In [5]:
def lgb_train(X_tr, y_tr, sw_tr, seed=42):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
        class_weight='balanced', objective='multiclass', num_class=3,
        n_jobs=-1, verbose=-1, random_state=seed)
    m.fit(X_tr, y_tr, sample_weight=sw_tr, callbacks=[lgb.log_evaluation(-1)])
    return m

def cat_train(X_tr, y_tr, sw_tr, seed=42):
    cw = [class_weights[0], class_weights[1], class_weights[2]]
    m = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
        loss_function='MultiClass', class_weights=cw,
        random_seed=seed, verbose=False, allow_writing_files=False)
    m.fit(X_tr, y_tr, sample_weight=sw_tr)
    return m

logo = LeaveOneGroupOut()
lopo_lgb, lopo_cat, lopo_blend = [], [], []
print('=== LOPO CV — v11 (v7c features + LGB+CAT blend, NO lag, NO SMOTE) ===')
for tr_idx, va_idx in logo.split(X_imp, y, groups):
    pid_val = groups.iloc[va_idx[0]]
    y_tr = y.iloc[tr_idx].values
    y_va = y.iloc[va_idx].values
    if len(set(y_va)) < 2:
        print(f'  Skip {pid_val}: only one class')
        continue
    X_tr = X_imp.iloc[tr_idx].values
    X_va = X_imp.iloc[va_idx].values
    sw_tr = sample_weights[tr_idx]

    m_lgb = lgb_train(X_tr, y_tr, sw_tr, seed=42)
    m_cat = cat_train(X_tr, y_tr, sw_tr, seed=42)
    p_lgb = m_lgb.predict_proba(X_va)
    p_cat = m_cat.predict_proba(X_va)
    p_blend = 0.5*p_lgb + 0.5*p_cat

    s_lgb   = balanced_accuracy_score(y_va, np.argmax(p_lgb, axis=1))
    s_cat   = balanced_accuracy_score(y_va, np.argmax(p_cat, axis=1))
    s_blend = balanced_accuracy_score(y_va, np.argmax(p_blend, axis=1))
    print(f'  Leave out {pid_val}: LGB={s_lgb:.4f}  CAT={s_cat:.4f}  BLEND={s_blend:.4f}  (n={len(va_idx)})')
    lopo_lgb.append(s_lgb)
    lopo_cat.append(s_cat)
    lopo_blend.append(s_blend)

print()
print(f'v11 LGB-only LOPO   = {np.mean(lopo_lgb):.4f} +/- {np.std(lopo_lgb):.4f}')
print(f'v11 CAT-only LOPO   = {np.mean(lopo_cat):.4f} +/- {np.std(lopo_cat):.4f}')
print(f'v11 BLEND LOPO      = {np.mean(lopo_blend):.4f} +/- {np.std(lopo_blend):.4f}')
print(f'v7c reference LOPO  = 0.5130')
print()
sanity_ok = abs(np.mean(lopo_lgb) - 0.5130) < 0.01
print(f'Sanity check (LGB-only ≈ 0.5130): {"PASS" if sanity_ok else "FAIL"}')

best_idx = int(np.argmax([np.mean(lopo_lgb), np.mean(lopo_cat), np.mean(lopo_blend)]))
best_name = ['LGB','CAT','BLEND'][best_idx]
print(f'Best on LOPO: {best_name}')


=== LOPO CV — v11 (v7c features + LGB+CAT blend, NO lag, NO SMOTE) ===
  Leave out 43JW: LGB=0.3599  CAT=0.5000  BLEND=0.3324  (n=93)
  Leave out C8Q6: LGB=0.4859  CAT=0.4507  BLEND=0.4859  (n=152)
  Leave out DT5C: LGB=0.3793  CAT=0.4782  BLEND=0.3793  (n=90)
  Leave out F1ZM: LGB=0.4963  CAT=0.4664  BLEND=0.4963  (n=137)
  Leave out HDS9: LGB=0.4209  CAT=0.2308  BLEND=0.3825  (n=135)
  Leave out P4DZ: LGB=0.3016  CAT=0.2778  BLEND=0.3016  (n=144)
  Leave out TPQI: LGB=0.6024  CAT=0.4756  BLEND=0.5676  (n=64)

v11 LGB-only LOPO   = 0.4352 +/- 0.0934
v11 CAT-only LOPO   = 0.4114 +/- 0.1011
v11 BLEND LOPO      = 0.4208 +/- 0.0900
v7c reference LOPO  = 0.5130

Sanity check (LGB-only ≈ 0.5130): FAIL
Best on LOPO: LGB


## Final ensemble — train all three, save 3 submissions

If LGB matches v7c (sanity passed) and BLEND beats v7c → submit BLEND.
If LGB matches v7c but BLEND ≤ v7c → blend doesn't help, do not submit.


In [6]:
SEEDS = [42, 7, 123]
n_test = len(X_test_imp)

all_lgb_proba = []
all_cat_proba = []

print('=== Final ensemble: 3 seeds x 5 folds (no SMOTE, no lag) ===')
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_lgb_test, fold_cat_test = [], []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_imp, y), 1):
        X_tr = X_imp.iloc[tr_idx].values
        y_tr = y.iloc[tr_idx].values
        X_va = X_imp.iloc[va_idx].values
        y_va = y.iloc[va_idx].values
        sw_tr = sample_weights[tr_idx]

        m_lgb = lgb_train(X_tr, y_tr, sw_tr, seed=seed)
        m_cat = cat_train(X_tr, y_tr, sw_tr, seed=seed)

        p_va_lgb = m_lgb.predict_proba(X_va)
        p_va_cat = m_cat.predict_proba(X_va)
        s_lgb = balanced_accuracy_score(y_va, np.argmax(p_va_lgb, axis=1))
        s_cat = balanced_accuracy_score(y_va, np.argmax(p_va_cat, axis=1))
        print(f'  Seed {seed} Fold {fold}: LGB={s_lgb:.4f}  CAT={s_cat:.4f}')

        fold_lgb_test.append(m_lgb.predict_proba(X_test_imp.values))
        fold_cat_test.append(m_cat.predict_proba(X_test_imp.values))

    all_lgb_proba.append(np.mean(fold_lgb_test, axis=0))
    all_cat_proba.append(np.mean(fold_cat_test, axis=0))

raw_lgb_proba = np.mean(all_lgb_proba, axis=0)
raw_cat_proba = np.mean(all_cat_proba, axis=0)
raw_blend     = 0.5*raw_lgb_proba + 0.5*raw_cat_proba

print()
print('Raw distributions:')
for name, p in [('LGB',raw_lgb_proba),('CAT',raw_cat_proba),('BLEND',raw_blend)]:
    pred = np.argmax(p, axis=1)
    print(f'  {name}: ', dict(Counter(pred)))


=== Final ensemble: 3 seeds x 5 folds (no SMOTE, no lag) ===
  Seed 42 Fold 1: LGB=0.8177  CAT=0.7595
  Seed 42 Fold 2: LGB=0.7247  CAT=0.7773
  Seed 42 Fold 3: LGB=0.8070  CAT=0.8386
  Seed 42 Fold 4: LGB=0.8133  CAT=0.7862
  Seed 42 Fold 5: LGB=0.8456  CAT=0.8517
  Seed 7 Fold 1: LGB=0.8321  CAT=0.8604
  Seed 7 Fold 2: LGB=0.8337  CAT=0.7855
  Seed 7 Fold 3: LGB=0.7910  CAT=0.7931
  Seed 7 Fold 4: LGB=0.8293  CAT=0.8107
  Seed 7 Fold 5: LGB=0.7394  CAT=0.7369
  Seed 123 Fold 1: LGB=0.8159  CAT=0.8462
  Seed 123 Fold 2: LGB=0.6566  CAT=0.6760
  Seed 123 Fold 3: LGB=0.8837  CAT=0.8630
  Seed 123 Fold 4: LGB=0.8076  CAT=0.8108
  Seed 123 Fold 5: LGB=0.6899  CAT=0.7679

Raw distributions:
  LGB:  {np.int64(2): 352, np.int64(0): 442, np.int64(1): 234}
  CAT:  {np.int64(0): 774, np.int64(2): 37, np.int64(1): 217}
  BLEND:  {np.int64(2): 228, np.int64(0): 570, np.int64(1): 230}


In [7]:
CALIB_ALPHA = 1.8

def calibrate_and_save(raw_proba, name):
    cal = raw_proba * (train_prior ** CALIB_ALPHA)
    cal = cal / cal.sum(axis=1, keepdims=True)
    preds = np.argmax(cal, axis=1).astype(int)
    sub = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds})
    fname = f'submission_v11_{name}.csv'
    sub.to_csv(fname, index=False)
    dist = dict(Counter(preds))
    print(f'  {name}: {fname}  dist={dist}')
    return preds

print('=== Calibrated submissions (alpha=1.8) ===')
preds_lgb   = calibrate_and_save(raw_lgb_proba, 'lgb')
preds_cat   = calibrate_and_save(raw_cat_proba, 'cat')
preds_blend = calibrate_and_save(raw_blend,     'blend')

print()
print(f'Train prior: 0={counts[0]/total*100:.1f}%  1={counts[1]/total*100:.1f}%  2={counts[2]/total*100:.1f}%')
print()
print(f'=== Decision summary ===')
print(f'  LGB   LOPO = {np.mean(lopo_lgb):.4f}')
print(f'  CAT   LOPO = {np.mean(lopo_cat):.4f}')
print(f'  BLEND LOPO = {np.mean(lopo_blend):.4f}')
print(f'  v7c reference = 0.5130')
print()
print(f'>> Submit only if BLEND LOPO > 0.5130. Otherwise stick with v7c.')


=== Calibrated submissions (alpha=1.8) ===
  lgb: submission_v11_lgb.csv  dist={np.int64(2): 776, np.int64(0): 197, np.int64(1): 55}
  cat: submission_v11_cat.csv  dist={np.int64(2): 977, np.int64(0): 51}
  blend: submission_v11_blend.csv  dist={np.int64(2): 903, np.int64(0): 123, np.int64(1): 2}

Train prior: 0=19.9%  1=8.1%  2=72.0%

=== Decision summary ===
  LGB   LOPO = 0.4352
  CAT   LOPO = 0.4114
  BLEND LOPO = 0.4208
  v7c reference = 0.5130

>> Submit only if BLEND LOPO > 0.5130. Otherwise stick with v7c.
